## **9-stimulus / 2-context swap task with anchors — RNN actor-critic**

Value matrix (fixed, no noise):

| stimulus | ctx 0 | ctx 1 | type        |
|----------|-------|-------|-------------|
| s0       | 0.0   | 1.0   | swap        |
| s1       | 0.5   | 0.5   | fixed (mid) |
| s2       | 1.0   | 0.0   | swap        |
| s3       | 0.0   | 1.0   | swap        |
| s4       | 1.0   | 0.0   | swap        |
| s5       | 0.0   | 0.0   | anchor low  |
| s6       | 0.0   | 0.0   | anchor low  |
| s7       | 1.0   | 1.0   | anchor high |
| s8       | 1.0   | 1.0   | anchor high |

4 swap stimuli (2 low→high, 2 high→low), 1 fixed mid, 2 anchor-low, 2 anchor-high.
The balanced 4-swap design gives identity–value correlation r = 0 across contexts,
so a cross-context decoder relying purely on stimulus identity scores at chance.

Author: patrick.mccarthy@dpag.ox.ac.uk

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from collections import defaultdict
from datetime import datetime
from pathlib import Path
import pickle

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical

from cxval.tasks import StimulusSequence, StateSequence
from cxval.envs import TaskEnv
from cxval.models import RNN, ActorCritic
from cxval.agents import Agent
from cxval.analysis import (
    filter_act_dict, mean_pairs, mean_offdiag,
    pairwise_decode, crosscontext_decode, generalisation_matrix,
    value_decode_within, value_decode_cross, value_gen_matrix,
    binary_value_decode_within, binary_value_decode_cross,
    plot_generalisation_heatmap,
)
from cxval.vis import STYLE

plt.style.use(STYLE)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 1. Task

In [ ]:
# fixed value matrix — no noise
# rows = stimuli (s0..s8), cols = contexts (c0, c1)
value_matrix = np.array([
    [0.0, 1.0],   # s0: swap  low → high
    [0.5, 0.5],   # s1: fixed mid
    [1.0, 0.0],   # s2: swap  high → low
    [0.0, 1.0],   # s3: swap  low → high
    [1.0, 0.0],   # s4: swap  high → low
    [0.0, 0.0],   # s5: anchor low
    [0.0, 0.0],   # s6: anchor low
    [1.0, 1.0],   # s7: anchor high
    [1.0, 1.0],   # s8: anchor high
], dtype=np.float32)

n_contexts = 2
n_stimuli  = 9
contexts   = ["c0", "c1"]
stimuli    = ["s0", "s1", "s2", "s3", "s4", "s5", "s6", "s7", "s8"]

# identity–value correlation across contexts (should be 0 for balanced swap design)
id_val_r = np.corrcoef(value_matrix[:, 0], value_matrix[:, 1])[0, 1]
print(f"Identity–value correlation across contexts: r = {id_val_r:.3f}")

# task timing
trials_per_phase   = 300
phases_per_context = 1
context_reps       = 30
stim_timesteps     = 5
reward_timesteps   = 3
iti_timesteps      = (3, 8)
seed               = 42

stim_seq = StimulusSequence(
    value_matrix=value_matrix,
    trials_per_phase=trials_per_phase,
    phases_per_context=phases_per_context,
    context_order="sequential",
    context_reps=context_reps,
)
stim_seq.generate(seed=seed)

state_seq = StateSequence(
    stimulus_sequence=stim_seq,
    value_matrix=value_matrix,
    stim_timesteps=stim_timesteps,
    reward_timesteps=reward_timesteps,
    iti_timesteps=iti_timesteps,
)
states, rewards, reward_availability = state_seq.generate(seed=seed)

n_trials = len(stim_seq.trial_contexts)
obs_dim  = states.shape[1] + 2

print(f"Value matrix:\n{value_matrix}")
print(f"Trials: {n_trials}  |  Sequence: {len(states)} timesteps  |  Obs dim: {obs_dim}")
# ── stimulus group metadata ──────────────────────────────────────────────
stim_group_info = [
    {"name": "swap (low\u2192high)", "short": "swap\n(lo\u2192hi)", "indices": [0, 3]},
    {"name": "mid",                  "short": "mid",               "indices": [1]},
    {"name": "swap (high\u2192low)", "short": "swap\n(hi\u2192lo)","indices": [2, 4]},
    {"name": "anchor (low)",         "short": "anchor\n(low)",     "indices": [5, 6]},
    {"name": "anchor (high)",        "short": "anchor\n(high)",    "indices": [7, 8]},
]


In [ ]:
# value matrix heatmap
fig, ax = plt.subplots(figsize=(3, 3))
im = ax.imshow(value_matrix, cmap="hot", vmin=0, vmax=1, aspect="equal")
for i in range(n_stimuli):
    for j in range(n_contexts):
        v = value_matrix[i, j]
        ax.text(j, i, f"{v:.1f}", ha="center", va="center",
                color="white" if v < 0.55 else "black", fontsize=11, fontweight="bold")
ax.set_yticks(range(n_stimuli)); ax.set_yticklabels(stimuli)
ax.set_xticks(range(n_contexts)); ax.set_xticklabels(contexts)
ax.set_ylabel("Stimulus"); ax.set_xlabel("Context")
ax.set_title("Value matrix")
plt.colorbar(im, ax=ax, shrink=0.8, label="Reward probability")
plt.tight_layout()
plt.show()

In [ ]:
# state sequence structure
trial_starts     = np.array([t["trial_start"] for t in state_seq.trial_structure])
context_at_trial = np.array([t["context"]     for t in state_seq.trial_structure])
ctx_change_idx   = np.where(np.diff(context_at_trial) != 0)[0] + 1
context_bnd_ts   = trial_starts[ctx_change_idx]

iti_indicator = (
    (states[:, n_contexts : n_contexts + n_stimuli].sum(axis=1) == 0) &
    (states[:, -1] == 0)
).astype(float)

vis_states = np.vstack([
    states.T,
    reward_availability[None, :],
    iti_indicator[None, :],
])
ylabels = (
    [f"ctx {c}" for c in contexts]
    + [f"stim {s}" for s in stimuli]
    + ["window_cue", "reward_avail", "ITI"]
)

ctx_colors = [plt.cm.Set2(v) for v in np.linspace(0, 0.75, max(n_contexts, 1))]

fig, ax = plt.subplots(figsize=(18, 4))
im = ax.imshow(vis_states, aspect="auto", cmap="hot", vmin=0, vmax=1, interpolation="nearest")
for t in context_bnd_ts:
    ax.axvline(t - 0.5, color="white", linewidth=2.5)
ax.set_yticks(range(len(ylabels)))
ax.set_yticklabels(ylabels)
ax.set_xlabel("Timestep")
ax.set_title("State sequence")
plt.colorbar(im, ax=ax, label="Activation")
plt.tight_layout()
plt.show()

## 2. Model

In [ ]:
lick_cost    = 0.0    # penalty for licking when no reward available, any timestep (0 = off)
policy_clip  = 0.05   # min probability for any action; prevents collapse to 0 (0 = off)

env = TaskEnv(
    states=states,
    reward_availability=reward_availability,
    reward_lick=1.0,
    reward_no_lick=0.0,
    reward_lick_miss=-1.0,
    lick_cost=lick_cost,
)

hidden_size    = 64
num_actions    = 2
recurrent_gain = 0.9

backbone     = RNN(input_size=obs_dim, hidden_size=hidden_size, output_size=1,
                   recurrent_gain=recurrent_gain)
actor_critic = ActorCritic(backbone=backbone, num_actions=num_actions,
                           policy_clip=policy_clip).to(device)
agent        = Agent(actor_critic, device=device)

n_params = sum(p.numel() for p in actor_critic.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")
print(f"policy_clip={policy_clip}  lick_cost={lick_cost}")

## 3. Train

In [ ]:
def compute_returns(rewards, bootstrap_value, gamma):
    returns, R = [], float(bootstrap_value)
    for r in reversed(rewards):
        R = r + gamma * R
        returns.append(R)
    returns.reverse()
    return returns

In [ ]:
n_episodes   = 1
bptt_len     = 10    # gradient horizon AND optimizer step frequency
update_every = 10    # how often to evaluate loss and accumulate gradients (must be <= bptt_len)
gamma        = 0.9
lr           = 9e-4
value_coef   = 0.5
entropy_coef = 0.01
grad_clip    = 1.0

optimizer = torch.optim.Adam(actor_critic.parameters(), lr=lr)

actor_losses    = []
critic_losses   = []
all_trial_data  = []
hidden_np_train = None

for episode in range(n_episodes):
    obs, _ = env.reset()
    hidden = None
    actor_critic.train()

    log_probs_buf, values_buf, rewards_buf, entropies_buf = [], [], [], []
    action_list, reward_list, info_list = [], [], []
    value_ts = []

    is_last = (episode == n_episodes - 1)
    if is_last:
        hidden_ts_list = []

    optimizer.zero_grad()
    t_in_window = 0
    n_opt_steps = 0
    done = False
    t    = 0

    while not done:
        obs_t  = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
        logits, value, hidden = actor_critic.step(obs_t, hidden)
        dist   = actor_critic.make_dist(logits)
        action = dist.sample()

        log_probs_buf.append(dist.log_prob(action))
        values_buf.append(value)
        entropies_buf.append(dist.entropy())
        value_ts.append(value.detach().item())

        if is_last:
            hidden_ts_list.append(hidden.detach().squeeze(0))

        action_list.append(action.item())
        obs, reward, done, _, info = env.step(action.item())
        rewards_buf.append(reward)
        reward_list.append(reward)
        info_list.append(info)
        t           += 1
        t_in_window += 1

        if t % update_every == 0 or done:
            if done:
                bootstrap_v = 0.0
            else:
                with torch.no_grad():
                    obs_next = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
                    _, bv, _ = actor_critic.step(obs_next, hidden)
                    bootstrap_v = bv.item()

            returns      = compute_returns(rewards_buf, bootstrap_v, gamma)
            log_probs_t  = torch.stack(log_probs_buf).squeeze(-1)
            values_t     = torch.stack(values_buf).squeeze(-1)
            returns_t    = torch.tensor(returns, dtype=torch.float32, device=device)
            entropy_mean = torch.stack(entropies_buf).mean()

            advantages = returns_t - values_t.detach()
            # only normalise when there is meaningful variance; with short windows
            # (e.g. pure-ITI chunks) std ≈ 0 and normalisation blows up the loss
            adv_std = advantages.std()
            if adv_std > 1e-4:
                advantages = (advantages - advantages.mean()) / (adv_std + 1e-8)
            else:
                advantages = advantages - advantages.mean()

            actor_loss  = -(log_probs_t * advantages).mean()
            critic_loss = F.mse_loss(values_t, returns_t)
            loss        = actor_loss + value_coef * critic_loss - entropy_coef * entropy_mean

            is_last_in_window = (t_in_window >= bptt_len) or done
            loss.backward(retain_graph=not is_last_in_window)

            actor_losses.append(actor_loss.item())
            critic_losses.append(critic_loss.item())

            log_probs_buf, values_buf, rewards_buf, entropies_buf = [], [], [], []

            if is_last_in_window:
                nn.utils.clip_grad_norm_(actor_critic.parameters(), grad_clip)
                optimizer.step()
                optimizer.zero_grad()
                n_opt_steps += 1
                hidden = hidden.detach()
                t_in_window = 0

    if is_last:
        hidden_np_train = np.array(torch.stack(hidden_ts_list).tolist(), dtype=np.float32)

    for ti, trial in enumerate(state_seq.trial_structure):
        rs, re = trial["reward_window"]
        all_trial_data.append({
            "global_trial":     episode * n_trials + ti,
            "context":          trial["context"],
            "stimulus":         trial["stimulus"],
            "reward_available": trial["reward_available"],
            "licked":           int(action_list[rs] == TaskEnv.LICK),
            "value_estimate":   float(np.mean(value_ts[rs:re])),
            "lick_count":       sum(1 for a in action_list[rs:re] if a == TaskEnv.LICK),
        })

    n_loss_evals = len(actor_losses)
    print(f"ep {episode+1}/{n_episodes} | "
          f"bptt_len={bptt_len} | update_every={update_every} | "
          f"loss evals: {n_loss_evals} | optimizer steps: {n_opt_steps}")

print(f"Training hidden states shape: {hidden_np_train.shape}")

## 4. Loss

In [ ]:
def smooth(x, w):
    x = np.array(x, dtype=float)
    if len(x) < w:
        return x, np.arange(len(x))
    return np.convolve(x, np.ones(w) / w, mode="valid"), np.arange(w - 1, len(x))

upd_w = max(1, min(50, len(actor_losses) // 10))

fig, ax = plt.subplots(figsize=(6, 3.5))
s_a, x_a = smooth(actor_losses,  upd_w)
s_c, x_c = smooth(critic_losses, upd_w)
ax.plot(x_a, s_a, label="actor")
ax.plot(x_c, s_c, label="critic", linestyle="--")
ax.set_title("Loss (per gradient update)")
ax.set_xlabel("Update step")
ax.legend()

# robust y-limits: clip spikes at 1st/99th percentile of smoothed values
all_smooth = np.concatenate([s_a, s_c])
ax.set_ylim(np.percentile(all_smooth, 1) - 0.05,
            np.percentile(all_smooth, 99) + 0.05)

plt.tight_layout()
plt.show()

## 5. Per-stimulus traces

In [ ]:
# ── inference run: one block per context, plasticity off ───────────────────
infer_stim_seq = StimulusSequence(
    value_matrix=value_matrix,
    trials_per_phase=trials_per_phase,
    phases_per_context=1,
    context_order="sequential",
    context_reps=1,
)
infer_stim_seq.generate(seed=seed + 1)

infer_state_seq = StateSequence(
    stimulus_sequence=infer_stim_seq,
    value_matrix=value_matrix,
    stim_timesteps=stim_timesteps,
    reward_timesteps=reward_timesteps,
    iti_timesteps=iti_timesteps,
)
infer_states, _, infer_reward_avail = infer_state_seq.generate(seed=seed + 1)

infer_env = TaskEnv(
    states=infer_states,
    reward_availability=infer_reward_avail,
    reward_lick=env.reward_lick,
    reward_no_lick=env.reward_no_lick,
    reward_lick_miss=env.reward_lick_miss,
    lick_cost=lick_cost,
)

actor_critic.eval()
agent.reset()
obs, _ = infer_env.reset()
infer_action_seq, infer_value_ts, infer_hidden_list = [], [], []
done = False
while not done:
    action, _, value = agent.act(obs)
    infer_action_seq.append(action)
    infer_value_ts.append(value.item())
    infer_hidden_list.append(agent.hidden.detach().squeeze(0))  # (hidden_size,)
    obs, _, done, _, _ = infer_env.step(action)

infer_hidden_np = np.array(torch.stack(infer_hidden_list).tolist(), dtype=np.float32)

infer_n_trials = len(infer_state_seq.trial_structure)
infer_trial_data = []
for ti, trial in enumerate(infer_state_seq.trial_structure):
    rs, re = trial["reward_window"]
    infer_trial_data.append({
        "global_trial":     n_trials + ti,
        "context":          trial["context"],
        "stimulus":         trial["stimulus"],
        "reward_available": trial["reward_available"],
        "licked":           int(infer_action_seq[rs] == TaskEnv.LICK),
        "value_estimate":   float(np.mean(infer_value_ts[rs:re])),
        "lick_count":       sum(1 for a in infer_action_seq[rs:re] if a == TaskEnv.LICK),
    })

infer_td = np.array([
    (d["global_trial"], d["context"], d["stimulus"],
     d["reward_available"], d["licked"], d["value_estimate"], d["lick_count"])
    for d in infer_trial_data
])
print(f"Inference: {infer_n_trials} trials, hidden states: {infer_hidden_np.shape}")

In [ ]:
td = np.array([
    (d["global_trial"], d["context"], d["stimulus"],
     d["reward_available"], d["licked"], d["value_estimate"], d["lick_count"])
    for d in all_trial_data
])
# cols: 0=global_trial 1=context 2=stimulus 3=reward_available
#       4=licked(1st rew step) 5=value_estimate 6=lick_count

total_n_trials = n_trials + infer_n_trials

# context block boundaries across training + inference
_ctx_arr       = np.array([t["context"] for t in state_seq.trial_structure])
_infer_ctx_arr = np.array([d["context"] for d in infer_trial_data])
_combined_ctx  = np.concatenate([_ctx_arr, _infer_ctx_arr])
_chg_idx         = np.where(np.diff(_combined_ctx) != 0)[0] + 1
ctx_block_starts = np.concatenate([[0], _chg_idx])
ctx_block_ends   = np.concatenate([_chg_idx, [total_n_trials]])
ctx_block_ctxs   = _combined_ctx[ctx_block_starts]

ctx_colors  = [plt.cm.Set2(v) for v in np.linspace(0, 0.75, max(n_contexts, 1))]
stim_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

smooth_w = max(1, min(3, n_trials // 10))

fig, axes = plt.subplots(4, 1, figsize=(15, 11),
                         gridspec_kw={"height_ratios": [3, 2.5, 2.5, 1], "hspace": 0.45})

# context background shading
for bs, be, ctx in zip(ctx_block_starts, ctx_block_ends, ctx_block_ctxs):
    for ax in axes[:3]:
        ax.axvspan(bs - 0.5, be - 0.5, alpha=0.15, color=ctx_colors[ctx], zorder=0, linewidth=0)

# plasticity boundary: dashed vertical line only
for ax in axes[:3]:
    ax.axvline(n_trials - 0.5, color="black", linewidth=1.2, linestyle="--", zorder=2)

# collect all smoothed value estimates for robust ylim
all_val_smoothed = []

# per-stimulus: training (solid) then inference (dashed, same colour)
for si in range(n_stimuli):
    c = stim_colors[si % len(stim_colors)]
    for data_td, ls in [(td, "-"), (infer_td, "--")]:
        mask        = data_td[:, 2] == si
        gt          = data_td[mask, 0].astype(int)
        val         = data_td[mask, 5]
        lick        = data_td[mask, 4]
        lick_rate_w = data_td[mask, 6] / reward_timesteps

        if smooth_w > 1 and len(val) >= smooth_w:
            sv  = np.convolve(val,         np.ones(smooth_w) / smooth_w, mode="valid")
            sl  = np.convolve(lick,        np.ones(smooth_w) / smooth_w, mode="valid")
            slr = np.convolve(lick_rate_w, np.ones(smooth_w) / smooth_w, mode="valid")
            xax = gt[smooth_w - 1:]
        else:
            sv, sl, slr, xax = val, lick, lick_rate_w, gt

        all_val_smoothed.append(sv)
        axes[0].plot(xax, sv,  color=c, linewidth=1.2, linestyle=ls)
        axes[1].plot(xax, sl,  color=c, linewidth=1.2, linestyle=ls)
        axes[2].plot(xax, slr, color=c, linewidth=1.2, linestyle=ls)

# robust ylim for value row: percentile-clipped but always includes [0, 1]
all_val_smoothed = np.concatenate(all_val_smoothed)
v_lo = min(0.0, np.percentile(all_val_smoothed, 1))
v_hi = max(1.0, np.percentile(all_val_smoothed, 99))
axes[0].set_ylim(v_lo - 0.05, v_hi + 0.05)

# ground-truth value lines per context
for si in range(n_stimuli):
    c = stim_colors[si % len(stim_colors)]
    for ci in range(n_contexts):
        axes[0].axhline(value_matrix[si, ci], color=c, linestyle=":", linewidth=0.8, alpha=0.5)

# "plasticity off" label
axes[0].text(n_trials + 0.5, 0.97, "← plasticity off",
             transform=axes[0].get_xaxis_transform(),
             fontsize=7, ha="left", va="top", color="dimgray")

stim_handles = [Line2D([0],[0], color=stim_colors[si % len(stim_colors)], label=stimuli[si])
                for si in range(n_stimuli)]
ctx_handles  = [Patch(facecolor=ctx_colors[c], alpha=0.6, label=contexts[c])
                for c in range(n_contexts)]
infer_handle = Line2D([0],[0], color="gray", linestyle="--", label="inference (no plasticity)")

axes[0].legend(handles=stim_handles + ctx_handles + [infer_handle],
               ncol=n_stimuli + n_contexts + 1, fontsize=8, loc="upper left")
axes[0].set_ylabel("Value estimate")
axes[0].set_title("Per-stimulus value estimate  (dotted = ground truth per context)")

axes[1].legend(handles=stim_handles, ncol=n_stimuli, fontsize=8, loc="upper left")
axes[1].set_ylabel("Lick prob.\n(1st rew. step)")
axes[1].set_ylim(-0.02, 1.02)
axes[1].set_title("Lick probability at first reward window timestep")

axes[2].set_ylabel("Lick rate\n(all rew. steps)")
axes[2].set_ylim(-0.02, 1.02)
axes[2].set_xlabel("Global trial index")
axes[2].set_title(f"Mean lick rate across all reward window timesteps ({reward_timesteps} ts)")

for ax in axes[:3]:
    ax.set_xlim(0, total_n_trials - 1)

# ── trial structure schematic ──────────────────────────────────────────────
ax = axes[3]
avg_iti   = state_seq.iti_durations.mean()
total_ts  = avg_iti + stim_timesteps + reward_timesteps
iti_frac  = avg_iti / total_ts
stim_frac = stim_timesteps / total_ts

for xstart, xend, color, label in [
    (0,                    iti_frac,             "lightgray", f"ITI (~{avg_iti:.0f} ts, jittered)"),
    (iti_frac,             iti_frac + stim_frac, "#4C72B0",   f"Stim ({stim_timesteps} ts)"),
    (iti_frac + stim_frac, 1.0,                  "#DD8452",   f"Reward window ({reward_timesteps} ts)"),
]:
    ax.barh(0, xend - xstart, left=xstart, height=0.5, color=color)
    ax.text((xstart + xend) / 2, 0, label, ha="center", va="center", fontsize=8,
            color="black" if color == "lightgray" else "white")

lick_x = iti_frac + stim_frac
ax.annotate("lick sampled\n(row 2 above)", xy=(lick_x, 0.25), xytext=(lick_x + 0.07, 1.0),
            arrowprops=dict(arrowstyle="->", color="black", lw=1.0), fontsize=7, ha="center")

ax.set_xlim(0, 1)
ax.set_ylim(-0.6, 1.6)
ax.axis("off")
ax.set_title(f"Trial structure (mean total ≈ {total_ts:.0f} ts)", fontsize=8, loc="left", pad=2)

plt.show()

In [ ]:
from scipy.stats import spearmanr
from matplotlib.lines import Line2D

# ── performance metrics (from inference / plasticity-off run) ──────────────
licked_arr     = np.array([d["licked"]           for d in infer_trial_data])
rew_avail      = np.array([d["reward_available"]  for d in infer_trial_data]).astype(bool)
stim_idx_arr   = np.array([d["stimulus"]          for d in infer_trial_data])
ctx_idx_arr    = np.array([d["context"]           for d in infer_trial_data])
lick_count_arr = np.array([d["lick_count"]        for d in infer_trial_data])

pct_reward_consumed      = licked_arr[rew_avail].mean()  * 100 if  rew_avail.any() else np.nan
false_alarm_rate         = licked_arr[~rew_avail].mean() * 100 if (~rew_avail).any() else np.nan
outcome_lick_proportion  = lick_count_arr.mean() / reward_timesteps * 100

stim_colors_p = plt.rcParams["axes.prop_cycle"].by_key()["color"]
ctx_markers   = ["o", "s", "^", "D", "v"]

lick_probs_flat, reward_probs_flat = [], []

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ax_sc = axes[0]

for si in range(n_stimuli):
    for ci in range(n_contexts):
        mask_sc = (stim_idx_arr == si) & (ctx_idx_arr == ci)
        if mask_sc.sum() == 0:
            continue
        lp = licked_arr[mask_sc].mean()
        rp = float(value_matrix[si, ci])
        lick_probs_flat.append(lp)
        reward_probs_flat.append(rp)
        ax_sc.scatter(rp, lp,
                      color=stim_colors_p[si % len(stim_colors_p)],
                      marker=ctx_markers[ci % len(ctx_markers)],
                      s=80, zorder=3)
        ax_sc.annotate(f"{stimuli[si]},{contexts[ci]}", (rp, lp),
                       textcoords="offset points", xytext=(5, 3), fontsize=7)

lick_probs_flat   = np.array(lick_probs_flat)
reward_probs_flat = np.array(reward_probs_flat)
r_sp = spearmanr(reward_probs_flat, lick_probs_flat)[0] if len(lick_probs_flat) > 1 else np.nan

ax_sc.plot([0, 1], [0, 1], "k--", linewidth=0.8, alpha=0.5, label="ideal")
ax_sc.set_xlim(-0.05, 1.05); ax_sc.set_ylim(-0.05, 1.05)
ax_sc.set_xlabel("Reward probability"); ax_sc.set_ylabel("Lick probability")
ax_sc.set_title(f"Lick–value calibration  (Spearman r = {r_sp:.3f})\n[inference / plasticity off]")

stim_h = [Line2D([0],[0], color=stim_colors_p[si % len(stim_colors_p)],
                 marker="o", linestyle="none", label=stimuli[si]) for si in range(n_stimuli)]
ctx_h  = [Line2D([0],[0], color="gray",
                 marker=ctx_markers[ci % len(ctx_markers)], linestyle="none",
                 label=contexts[ci]) for ci in range(n_contexts)]
ax_sc.legend(handles=stim_h + ctx_h, fontsize=8, ncol=2, loc="upper left")

ax_bar = axes[1]
bars = ax_bar.bar(
    ["Reward\nconsumed (%)", "False alarm\nrate (%)", "Outcome lick\nproportion (%)"],
    [pct_reward_consumed, false_alarm_rate, outcome_lick_proportion],
    color=["steelblue", "salmon", "gray"], width=0.5,
)
ax_bar.set_ylim(0, 110); ax_bar.set_ylabel("%")
ax_bar.set_title("Performance summary  [inference / plasticity off]")
for bar, val in zip(bars, [pct_reward_consumed, false_alarm_rate, outcome_lick_proportion]):
    if not np.isnan(val):
        ax_bar.text(bar.get_x() + bar.get_width() / 2, val + 1,
                    f"{val:.1f}%", ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import spearmanr

# ── inference-phase arrays ─────────────────────────────────────────────────
inf_stim  = np.array([d["stimulus"]         for d in infer_trial_data])
inf_ctx   = np.array([d["context"]          for d in infer_trial_data])
inf_lick  = np.array([d["licked"]           for d in infer_trial_data])          # 1st reward step
inf_lkct  = np.array([d["lick_count"]       for d in infer_trial_data])          # all reward steps
inf_vest  = np.array([d["value_estimate"]   for d in infer_trial_data])          # mean over reward window
inf_ravail = np.array([d["reward_available"] for d in infer_trial_data]).astype(bool)

# TD error: actual reward at 1st reward step minus mean value estimate over window
def _actual_r(licked, ravail):
    if licked:
        base  = env.reward_lick if ravail else env.reward_lick_miss
        extra = lick_cost if (not ravail and lick_cost != 0) else 0.0
        return base + extra
    return env.reward_no_lick

inf_tderr = np.array([_actual_r(l, ra) for l, ra in zip(inf_lick.tolist(), inf_ravail.tolist())]) - inf_vest

# per-stim per-context means
val_sc  = np.full((n_stimuli, n_contexts), np.nan)
lick_sc = np.full((n_stimuli, n_contexts), np.nan)
lkrt_sc = np.full((n_stimuli, n_contexts), np.nan)
td_sc   = np.full((n_stimuli, n_contexts), np.nan)
for si in range(n_stimuli):
    for ci in range(n_contexts):
        m = (inf_stim == si) & (inf_ctx == ci)
        if m.sum() == 0:
            continue
        val_sc[si, ci]  = inf_vest[m].mean()
        lick_sc[si, ci] = inf_lick[m].mean()
        lkrt_sc[si, ci] = inf_lkct[m].mean() / reward_timesteps
        td_sc[si, ci]   = inf_tderr[m].mean()

_sc = [plt.rcParams["axes.prop_cycle"].by_key()["color"][si % 10] for si in range(n_stimuli)]
_cm = ["o", "s", "^", "D", "v"]
x   = np.arange(n_stimuli)
w   = 0.35

# ── Figure 1: per-stimulus per-context grouped bars ────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True,
                         gridspec_kw={"hspace": 0.45})

row_data = [
    (val_sc,  "Value estimate\n(mean over reward window,\nsteps rs : re)",
     "Value estimates  [timesteps: mean over all reward-window steps]"),
    (lick_sc, "Lick probability\n(1st reward-window step, rs)",
     "Lick probability at 1st reward-window step  [timestep: rs only]"),
    (lkrt_sc, f"Outcome lick proportion\n(mean licks / {reward_timesteps} reward steps)",
     f"Outcome lick proportion  [timesteps: all {reward_timesteps} reward-window steps]"),
]

for ax, (data, ylabel, title) in zip(axes, row_data):
    for ci in range(n_contexts):
        offset = (ci - (n_contexts - 1) / 2) * w
        ax.bar(x + offset, data[:, ci], width=w,
               color=ctx_colors[ci], alpha=0.85, label=contexts[ci], zorder=2)
        # ground-truth reward probability overlaid on every panel
        for si in range(n_stimuli):
            ax.plot([x[si] + offset - w / 2, x[si] + offset + w / 2],
                    [value_matrix[si, ci]] * 2,
                    color="black", linewidth=2.0, zorder=5)

    ax.set_ylim(-0.1, 1.1)
    ax.axhline(0, color="gray", linewidth=0.5, linestyle=":")
    ax.axhline(1, color="gray", linewidth=0.5, linestyle=":")
    ax.set_ylabel(ylabel, fontsize=8)
    ax.set_title(title + "\n(black lines = ground-truth reward probability)", fontsize=9)
    ax.legend(fontsize=8)

axes[-1].set_xticks(x)
axes[-1].set_xticklabels(stimuli)
axes[-1].set_xlabel("Stimulus")
fig.suptitle("Per-stimulus per-context inference summary  [plasticity off]", y=1.01)
plt.tight_layout()
plt.show()

# ── Figure 2: combined performance metrics ─────────────────────────────────
lp_flat, rp_flat, si_flat, ci_flat = [], [], [], []
for si in range(n_stimuli):
    for ci in range(n_contexts):
        m = (inf_stim == si) & (inf_ctx == ci)
        if m.sum() == 0:
            continue
        lp_flat.append(float(inf_lick[m].mean()))
        rp_flat.append(float(value_matrix[si, ci]))
        si_flat.append(si); ci_flat.append(ci)

lp_flat = np.array(lp_flat); rp_flat = np.array(rp_flat)
r_sp = spearmanr(rp_flat, lp_flat)[0] if len(lp_flat) > 1 else np.nan

pct_consumed = inf_lick[inf_ravail].mean()  * 100 if  inf_ravail.any() else np.nan
pct_fa       = inf_lick[~inf_ravail].mean() * 100 if (~inf_ravail).any() else np.nan
pct_lkrt     = inf_lkct.mean() / reward_timesteps * 100

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

# -- calibration scatter --
ax = axes[0]
for lp, rp, si, ci in zip(lp_flat, rp_flat, si_flat, ci_flat):
    ax.scatter(rp, lp, color=_sc[si], marker=_cm[ci % len(_cm)], s=80, zorder=3)
    ax.annotate(f"{stimuli[si]},{contexts[ci]}", (rp, lp),
                textcoords="offset points", xytext=(5, 3), fontsize=6)
ax.plot([0, 1], [0, 1], "k--", linewidth=0.8, alpha=0.5, label="ideal")
ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
ax.set_xlabel("Reward probability"); ax.set_ylabel("Lick probability\n(1st reward step)")
ax.set_title(f"Lick–value calibration\nSpearman r = {r_sp:.3f}")
stim_h = [Line2D([0],[0], color=_sc[si], marker="o", linestyle="none", label=stimuli[si])
          for si in range(n_stimuli)]
ctx_h  = [Line2D([0],[0], color="gray", marker=_cm[ci], linestyle="none", label=contexts[ci])
          for ci in range(n_contexts)]
ax.legend(handles=stim_h + ctx_h, fontsize=7, ncol=3)

# -- TD error per stim (mean across contexts) --
ax = axes[1]
td_per_stim = np.nanmean(td_sc, axis=1)
bars = ax.bar(stimuli, td_per_stim, color=_sc, width=0.6)
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax.set_ylabel("Mean TD error  (r − V̂)")
ax.set_xlabel("Stimulus")
ax.set_title("TD error per stimulus\n(mean across contexts  [inference])\n"
             "TD = actual reward at 1st reward step − mean value estimate over window")
for bar, val in zip(bars, td_per_stim):
    if not np.isnan(val):
        va, yo = ("bottom", 0.01) if val >= 0 else ("top", -0.01)
        ax.text(bar.get_x() + bar.get_width() / 2, val + yo,
                f"{val:.2f}", ha="center", va=va, fontsize=8)

# -- performance bar --
ax = axes[2]
perf_labels = ["Reward\nconsumed (%)", "False alarm\nrate (%)", "Outcome lick\nproportion (%)"]
perf_vals   = [pct_consumed, pct_fa, pct_lkrt]
perf_colors = ["steelblue", "salmon", "gray"]
bars = ax.bar(perf_labels, perf_vals, color=perf_colors, width=0.5)
ax.set_ylim(0, 110); ax.set_ylabel("%")
ax.set_title("Performance summary  [inference / plasticity off]")
for bar, val in zip(bars, perf_vals):
    if not np.isnan(val):
        ax.text(bar.get_x() + bar.get_width() / 2, val + 1,
                f"{val:.1f}%", ha="center", va="bottom", fontsize=10)

fig.suptitle("Combined performance metrics  [inference / plasticity off]", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── per-period lick rates (ITI / stim / reward window) ────────────────────
# Uses raw infer_action_seq so all timesteps within each window are counted.
# Reward window is split by whether reward was available (rewarded vs unrewarded trial).

_iti_rates, _stim_rates, _rew_r_rates, _rew_u_rates = [], [], [], []
_pc_ctx, _pc_stim, _pc_ravail = [], [], []

for trial in infer_state_seq.trial_structure:
    iti_s,  iti_e  = trial["iti_window"]
    stim_s, stim_e = trial["stim_window"]
    rew_s,  rew_e  = trial["reward_window"]

    def _lick_rate(actions, s, e):
        seg = [a == TaskEnv.LICK for a in actions[s:e]]
        return float(np.mean(seg)) if seg else np.nan

    _iti_rates.append(_lick_rate(infer_action_seq, iti_s,  iti_e))
    _stim_rates.append(_lick_rate(infer_action_seq, stim_s, stim_e))
    rew_rate = _lick_rate(infer_action_seq, rew_s, rew_e)
    _rew_r_rates.append(rew_rate if trial["reward_available"] else np.nan)
    _rew_u_rates.append(rew_rate if not trial["reward_available"] else np.nan)

    _pc_ctx.append(trial["context"])
    _pc_stim.append(trial["stimulus"])
    _pc_ravail.append(trial["reward_available"])

_pc_ctx    = np.array(_pc_ctx)
_pc_stim   = np.array(_pc_stim)
_pc_ravail = np.array(_pc_ravail, dtype=bool)

# per-stim per-context means for each period
def _per_stim_ctx(rates):
    rates = np.array(rates, dtype=float)
    out = np.full((n_stimuli, n_contexts), np.nan)
    for si in range(n_stimuli):
        for ci in range(n_contexts):
            m = (_pc_stim == si) & (_pc_ctx == ci)
            vals = rates[m]
            if np.any(~np.isnan(vals)):
                out[si, ci] = np.nanmean(vals)
    return out

_iti_sc   = _per_stim_ctx(_iti_rates)
_stim_sc  = _per_stim_ctx(_stim_rates)
_rewr_sc  = _per_stim_ctx(_rew_r_rates)
_rewu_sc  = _per_stim_ctx(_rew_u_rates)

# ── figure A: overall period summary ──────────────────────────────────────
overall = {
    "ITI": np.nanmean(_iti_rates),
    "Stim": np.nanmean(_stim_rates),
    "Reward\n(rewarded trials)": np.nanmean(_rew_r_rates),
    "Reward\n(unrewarded trials)": np.nanmean(_rew_u_rates),
}
fig, (ax_ov, ax_ctx) = plt.subplots(1, 2, figsize=(11, 4))

bars = ax_ov.bar(list(overall), list(overall.values()),
                 color=["lightgray", "#4C72B0", "#55A868", "#DD8452"], width=0.55)
ax_ov.axhline(0.5, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
ax_ov.set_ylim(0, 1.05); ax_ov.set_ylabel("Mean lick rate")
ax_ov.set_title("Overall lick rate by trial period\n[inference / plasticity off]")
for bar, val in zip(bars, overall.values()):
    if not np.isnan(val):
        ax_ov.text(bar.get_x() + bar.get_width() / 2, val + 0.02,
                   f"{val:.2f}", ha="center", va="bottom", fontsize=10)

# split by context — use _xp to avoid clobbering the stim x-axis used below
_xp = np.arange(4)
for ci in range(n_contexts):
    m = _pc_ctx == ci
    ctx_ov = [
        np.nanmean(np.array(_iti_rates,   float)[m]),
        np.nanmean(np.array(_stim_rates,  float)[m]),
        np.nanmean(np.array(_rew_r_rates, float)[_pc_ravail  & m]),
        np.nanmean(np.array(_rew_u_rates, float)[~_pc_ravail & m]),
    ]
    offset = (ci - (n_contexts - 1) / 2) * 0.3
    ax_ctx.bar(_xp + offset, ctx_ov, width=0.28,
               color=ctx_colors[ci], alpha=0.85, label=contexts[ci])

ax_ctx.set_xticks(_xp)
ax_ctx.set_xticklabels(["ITI", "Stim", "Reward\n(rewarded)", "Reward\n(unrewarded)"])
ax_ctx.axhline(0.5, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
ax_ctx.set_ylim(0, 1.05); ax_ctx.set_ylabel("Mean lick rate")
ax_ctx.set_title("Lick rate by period and context\n[inference / plasticity off]")
ax_ctx.legend(fontsize=9)

fig.suptitle("Period-specific lick rates — check for always-licking policy", y=1.02)
plt.tight_layout()
plt.show()

# ── figure B: per-stim per-context grouped bars for each period ────────────
_xs = np.arange(n_stimuli)   # stim x-axis (separate from _xp above)

fig, axes = plt.subplots(4, 1, figsize=(14, 13), sharex=True,
                         gridspec_kw={"hspace": 0.5})

_period_rows = [
    (_iti_sc,  "ITI lick rate\n(all ITI timesteps)",
     "ITI period  [all timesteps]  —  should be ~0 if policy is selective"),
    (_stim_sc, "Stim lick rate\n(all stim timesteps)",
     f"Stimulus period  [all {stim_timesteps} timesteps]  —  should be ~0 if not pre-licking"),
    (_rewr_sc, "Reward-window lick rate\n(rewarded trials)",
     f"Reward window — rewarded trials  [{reward_timesteps} timesteps]"),
    (_rewu_sc, "Reward-window lick rate\n(unrewarded trials)",
     f"Reward window — unrewarded trials  [{reward_timesteps} timesteps]"),
]

for ax, (data, ylabel, title) in zip(axes, _period_rows):
    for ci in range(n_contexts):
        offset = (ci - (n_contexts - 1) / 2) * w
        ax.bar(_xs + offset, data[:, ci], width=w,
               color=ctx_colors[ci], alpha=0.85, label=contexts[ci], zorder=2)
        # GT reward probability for reference
        for si in range(n_stimuli):
            ax.plot([_xs[si] + offset - w / 2, _xs[si] + offset + w / 2],
                    [value_matrix[si, ci]] * 2,
                    color="black", linewidth=2.0, zorder=5)
    ax.set_ylim(-0.05, 1.1)
    ax.axhline(0, color="gray", linewidth=0.5, linestyle=":")
    ax.set_ylabel(ylabel, fontsize=8)
    ax.set_title(title + "\n(black lines = GT reward probability)", fontsize=9)
    ax.legend(fontsize=8)

axes[-1].set_xticks(_xs)
axes[-1].set_xticklabels(stimuli)
axes[-1].set_xlabel("Stimulus")
fig.suptitle("Per-stimulus lick rates across all trial periods  [inference / plasticity off]",
             y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import spearmanr

# ── behavioural learning curves ────────────────────────────────────────────
td_gtr    = np.array([d["global_trial"]     for d in all_trial_data])
td_lick   = np.array([d["licked"]           for d in all_trial_data])
td_stim   = np.array([d["stimulus"]         for d in all_trial_data])
td_ctx    = np.array([d["context"]          for d in all_trial_data])
td_ravail = np.array([d["reward_available"] for d in all_trial_data])
td_vest   = np.array([d["value_estimate"]   for d in all_trial_data])

# actual reward received at first reward-window step (post-hoc from stored fields)
def _actual_reward(licked, reward_available):
    if licked:
        base  = env.reward_lick if reward_available else env.reward_lick_miss
        extra = lick_cost if (not reward_available and lick_cost != 0) else 0.0
        return base + extra
    return env.reward_no_lick

td_errors = np.array([_actual_reward(l, ra)
                       for l, ra in zip(td_lick.tolist(), td_ravail.tolist())]) - td_vest

# global block: every trials_per_phase consecutive trials
global_ph = td_gtr // trials_per_phase
n_gph     = int(global_ph.max()) + 1

# context of each global block (consistent within block for sequential ordering)
gph_ctx = np.zeros(n_gph, dtype=int)
for d, gp in zip(all_trial_data, global_ph):
    gph_ctx[gp] = d["context"]

# context shading boundaries (in block units)
_chg = np.where(np.diff(gph_ctx) != 0)[0] + 1 if n_gph > 1 else np.array([], dtype=int)
gph_blk_starts = np.concatenate([[0], _chg])
gph_blk_ends   = np.concatenate([_chg, [n_gph]])
gph_blk_ctxs   = gph_ctx[gph_blk_starts]

# lick-value calibration Spearman r per global block
calib_r = np.full(n_gph, np.nan)
for gp in range(n_gph):
    ci = gph_ctx[gp]
    lp_flat, rp_flat = [], []
    for si in range(n_stimuli):
        mask = (global_ph == gp) & (td_stim == si)
        if mask.sum() == 0:
            continue
        lp_flat.append(float(td_lick[mask].mean()))
        rp_flat.append(float(value_matrix[si, ci]))
    if len(lp_flat) > 2:
        calib_r[gp] = spearmanr(rp_flat, lp_flat)[0]

# TD error per stimulus per global block
td_phase_stim = np.full((n_gph, n_stimuli), np.nan)
for gp in range(n_gph):
    for si in range(n_stimuli):
        mask = (global_ph == gp) & (td_stim == si)
        if mask.sum() > 0:
            td_phase_stim[gp, si] = td_errors[mask].mean()

gph_axis    = np.arange(n_gph)
stim_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True,
                          gridspec_kw={"hspace": 0.35})

for bs, be, ctx in zip(gph_blk_starts, gph_blk_ends, gph_blk_ctxs):
    for ax in axes:
        ax.axvspan(bs - 0.5, be - 0.5, alpha=0.15, color=ctx_colors[ctx], zorder=0, linewidth=0)

axes[0].plot(gph_axis, calib_r, color="black", linewidth=1.5, marker="o", ms=4)
axes[0].axhline(0, color="gray", linestyle="--", linewidth=0.8)
axes[0].axhline(1, color="gray", linestyle=":", linewidth=0.8, alpha=0.5)
axes[0].set_ylabel("Spearman r")
axes[0].set_ylim(-1.05, 1.05)
axes[0].set_title("Lick–value calibration per context block  (training)")
ctx_handles = [Patch(facecolor=ctx_colors[c], alpha=0.6, label=contexts[c])
               for c in range(n_contexts)]
axes[0].legend(handles=ctx_handles, fontsize=8, loc="lower right")

for si in range(n_stimuli):
    axes[1].plot(gph_axis, td_phase_stim[:, si],
                 color=stim_colors[si % len(stim_colors)],
                 linewidth=1.2, marker="o", ms=3, label=stimuli[si])
axes[1].axhline(0, color="gray", linestyle="--", linewidth=0.8)
axes[1].set_ylabel("Mean TD error  (r − V̂)")
axes[1].set_xlabel("Global context block")
axes[1].set_title("TD error per stimulus per context block  (training)")
stim_handles = [Line2D([0],[0], color=stim_colors[si % len(stim_colors)], label=stimuli[si])
                for si in range(n_stimuli)]
axes[1].legend(handles=stim_handles, ncol=n_stimuli, fontsize=8)

plt.tight_layout()
plt.show()

## 6. Post-training activations

In [ ]:
def t2np(tensor):
    return np.array(tensor.detach().cpu().tolist(), dtype=np.float32)

actor_critic.eval()
obs, _ = env.reset()
agent.reset()

obs_seq, action_seq = [], []
done = False
while not done:
    obs_seq.append(obs.copy())
    action, _, _ = agent.act(obs)
    action_seq.append(action)
    obs, _, done, _, _ = env.step(action)

with torch.no_grad():
    obs_t = torch.FloatTensor(np.stack(obs_seq)).unsqueeze(0).to(device)
    _, _, hidden_all = actor_critic(obs_t)
hidden_np = t2np(hidden_all.squeeze(0))   # (T, hidden_size)

# trial metadata
context_arr          = np.array([t["context"]          for t in state_seq.trial_structure])
stimulus_arr         = np.array([t["stimulus"]         for t in state_seq.trial_structure])
reward_available_arr = np.array([t["reward_available"] for t in state_seq.trial_structure])

_cc = defaultdict(int)
phase_arr = np.zeros(n_trials, dtype=int)
for i, ctx in enumerate(context_arr):
    phase_arr[i] = _cc[ctx] // trials_per_phase
    _cc[ctx] += 1

action_arr   = np.array([action_seq[t["reward_window"][0]] for t in state_seq.trial_structure])
rewarded_arr = (action_arr == TaskEnv.LICK) & reward_available_arr

stim_hidden   = np.stack([hidden_np[t["stim_window"][0]   : t["stim_window"][1]]   for t in state_seq.trial_structure])
reward_hidden = np.stack([hidden_np[t["reward_window"][0] : t["reward_window"][1]] for t in state_seq.trial_structure])

activations = {
    "hidden_states":    hidden_np,
    "stim_hidden":      stim_hidden,
    "reward_hidden":    reward_hidden,
    "context":          context_arr,
    "stimulus":         stimulus_arr,
    "phase":            phase_arr,
    "reward_available": reward_available_arr,
    "action":           action_arr,
    "rewarded":         rewarded_arr,
    "trial_structure":  state_seq.trial_structure,
}
print(f"hidden_states : {hidden_np.shape}")
print(f"stim_hidden   : {stim_hidden.shape}")
print(f"phases present: {np.unique(phase_arr)}")

In [ ]:
# activation heatmaps — mean hidden state per epoch, per stimulus
iti_hidden_mean = np.array([
    hidden_np[t["iti_window"][0]:t["iti_window"][1]].mean(0)
    if t["iti_window"][1] > t["iti_window"][0] else np.zeros(hidden_np.shape[1])
    for t in state_seq.trial_structure
])   # (n_trials, H)

stim_hidden_mean   = activations["stim_hidden"].mean(1)    # (n_trials, H)
reward_hidden_mean = activations["reward_hidden"].mean(1)  # (n_trials, H)

epochs_data = [
    ("ITI",    iti_hidden_mean),
    ("Stim",   stim_hidden_mean),
    ("Reward", reward_hidden_mean),
]

all_vals = np.concatenate([d.ravel() for _, d in epochs_data])
vmin_act, vmax_act = np.percentile(all_vals, 2), np.percentile(all_vals, 98)

fig, axes = plt.subplots(n_stimuli, 3, figsize=(12, 2.5 * n_stimuli), sharey=True)

for si in range(n_stimuli):
    mask     = stimulus_arr == si
    ctx_this = context_arr[mask]

    chg        = np.where(np.diff(ctx_this) != 0)[0] + 1 if ctx_this.size > 1 else np.array([], dtype=int)
    blk_starts = np.concatenate([[0], chg])
    blk_ends   = np.concatenate([chg, [mask.sum()]])
    blk_ctxs   = ctx_this[blk_starts]

    for col, (epoch_name, epoch_data) in enumerate(epochs_data):
        ax   = axes[si, col]
        data = epoch_data[mask]

        im = ax.imshow(data.T, aspect="auto", cmap="viridis",
                       vmin=vmin_act, vmax=vmax_act, interpolation="nearest")

        for bs, be, ctx in zip(blk_starts, blk_ends, blk_ctxs):
            ax.axvspan(bs - 0.5, be - 0.5, alpha=0.2,
                       color=ctx_colors[ctx], zorder=2, linewidth=0)

        if si == 0:
            ax.set_title(epoch_name, fontsize=10)
        if col == 0:
            ax.set_ylabel(f"{stimuli[si]}\nUnit", fontsize=8)
        if si == n_stimuli - 1:
            ax.set_xlabel("Trial", fontsize=8)
        ax.tick_params(labelsize=7)

fig.colorbar(im, ax=axes, shrink=0.5, label="Activation", pad=0.02)

ctx_handles = [Patch(facecolor=ctx_colors[c], alpha=0.7, label=contexts[c])
               for c in range(n_contexts)]
axes[0, -1].legend(handles=ctx_handles, loc="upper right", fontsize=7, framealpha=0.7)

plt.suptitle("Mean hidden-state activations per period (rows = stimuli)", y=1.01, fontsize=11)
plt.tight_layout()
plt.show()

## 7. Decoding — post-training

In [ ]:
pooling = "average"
n_folds = 5

# stimulus groups used for binary SVM decoding
# (also used by a0000023, defined here so cells run in order)
stim_groups  = {"all": None, "swap": [0, 2, 3, 4], "anchor": [5, 6, 7, 8]}
group_colors = {"all": "black", "swap": "tomato", "anchor": "steelblue"}

# stimulus identity generalisation
stim_acc       = pairwise_decode(activations, period="stim", pooling=pooling, n_folds=n_folds)
cross_stim_acc = crosscontext_decode(activations, period="stim", pooling=pooling)
gen_stim       = generalisation_matrix(stim_acc, cross_stim_acc)

# Ridge regression value generalisation
r_within = value_decode_within(activations, period="stim", pooling=pooling,
                               value_matrix=value_matrix, n_folds=n_folds)
r_cross  = value_decode_cross(activations,  period="stim", pooling=pooling,
                              value_matrix=value_matrix)
gm_val   = value_gen_matrix(r_within, r_cross)

# binary SVM value generalisation — all / swap / anchor
bin_gm = {}
for gname, gidx in stim_groups.items():
    act_g = (filter_act_dict(activations, np.isin(activations["stimulus"], gidx))
             if gidx is not None else activations)
    bw = binary_value_decode_within(act_g, period="stim", pooling=pooling,
                                    value_matrix=value_matrix, n_folds=n_folds)
    bc = binary_value_decode_cross(act_g,  period="stim", pooling=pooling,
                                   value_matrix=value_matrix)
    gm = bc.copy(); np.fill_diagonal(gm, bw)
    bin_gm[gname] = gm

cell_size = 1.1

# ── figure 1: stim identity + Ridge value ─────────────────────────────────
fig, axes = plt.subplots(1, 2,
                         figsize=(cell_size * n_contexts * 2 + 1, cell_size * n_contexts),
                         squeeze=False)
plot_generalisation_heatmap(axes[0, 0], gen_stim, contexts,
                            vmin=0, vmax=1, cmap="seismic",
                            colorbar_label="Accuracy",
                            title="Stim identity  (white = chance)")
plot_generalisation_heatmap(axes[0, 1], gm_val, contexts,
                            vmin=-1, vmax=1, cmap="seismic",
                            colorbar_label="Pearson r",
                            title="Value (Ridge)  (white = r=0)")
fig.suptitle(f"Post-training decoding  ·  stim period  ·  {pooling} pooling", y=1.04)
plt.tight_layout()
plt.show()

# ── figure 2: binary SVM by stimulus group ─────────────────────────────────
fig, axes = plt.subplots(1, len(stim_groups),
                         figsize=(cell_size * n_contexts * len(stim_groups) + 1,
                                  cell_size * n_contexts),
                         squeeze=False)
for ax, (gname, gm) in zip(axes[0], bin_gm.items()):
    plot_generalisation_heatmap(ax, gm, contexts,
                                vmin=0, vmax=1, cmap="seismic",
                                colorbar_label="Accuracy",
                                title=f"Binary SVM — {gname}  (chance = 0.5)")
fig.suptitle(
    f"Post-training binary SVM  ·  stim period  ·  {pooling} pooling\n"
    f"low (v=0) vs high (v=1), all identities pooled within group",
    y=1.06,
)
plt.tight_layout()
plt.show()

## 7b. Decoding — inference activations (plasticity off)

In [ ]:
# ── build infer_activations from hidden states collected during inference ──
infer_context_arr          = np.array([t["context"]          for t in infer_state_seq.trial_structure])
infer_stimulus_arr         = np.array([t["stimulus"]         for t in infer_state_seq.trial_structure])
infer_reward_available_arr = np.array([t["reward_available"] for t in infer_state_seq.trial_structure])

infer_stim_hidden   = np.stack([infer_hidden_np[t["stim_window"][0]   : t["stim_window"][1]]
                                 for t in infer_state_seq.trial_structure])
infer_reward_hidden = np.stack([infer_hidden_np[t["reward_window"][0] : t["reward_window"][1]]
                                 for t in infer_state_seq.trial_structure])

infer_activations = {
    "hidden_states":    infer_hidden_np,
    "stim_hidden":      infer_stim_hidden,
    "reward_hidden":    infer_reward_hidden,
    "context":          infer_context_arr,
    "stimulus":         infer_stimulus_arr,
    "reward_available": infer_reward_available_arr,
    "trial_structure":  infer_state_seq.trial_structure,
}
print(f"infer stim_hidden   : {infer_stim_hidden.shape}")
print(f"infer reward_hidden : {infer_reward_hidden.shape}")

# ── decoding ───────────────────────────────────────────────────────────────
# stim identity
infer_stim_acc       = pairwise_decode(infer_activations, period="stim",
                                       pooling=pooling, n_folds=n_folds)
infer_cross_stim_acc = crosscontext_decode(infer_activations, period="stim", pooling=pooling)
infer_gen_stim       = generalisation_matrix(infer_stim_acc, infer_cross_stim_acc)

# Ridge value
infer_r_within = value_decode_within(infer_activations, period="stim", pooling=pooling,
                                     value_matrix=value_matrix, n_folds=n_folds)
infer_r_cross  = value_decode_cross(infer_activations,  period="stim", pooling=pooling,
                                    value_matrix=value_matrix)
infer_gm_val   = value_gen_matrix(infer_r_within, infer_r_cross)

# binary SVM by group
infer_bin_gm = {}
for gname, gidx in stim_groups.items():
    act_g = (filter_act_dict(infer_activations, np.isin(infer_activations["stimulus"], gidx))
             if gidx is not None else infer_activations)
    bw = binary_value_decode_within(act_g, period="stim", pooling=pooling,
                                    value_matrix=value_matrix, n_folds=n_folds)
    bc = binary_value_decode_cross(act_g, period="stim", pooling=pooling,
                                   value_matrix=value_matrix)
    gm = bc.copy(); np.fill_diagonal(gm, bw)
    infer_bin_gm[gname] = gm

# ── figure 1: stim identity + Ridge value ─────────────────────────────────
fig, axes = plt.subplots(1, 2,
                         figsize=(cell_size * n_contexts * 2 + 1, cell_size * n_contexts),
                         squeeze=False)
plot_generalisation_heatmap(axes[0, 0], infer_gen_stim, contexts,
                            vmin=0, vmax=1, cmap="seismic",
                            colorbar_label="Accuracy",
                            title="Stim identity  (white = chance)")
plot_generalisation_heatmap(axes[0, 1], infer_gm_val, contexts,
                            vmin=-1, vmax=1, cmap="seismic",
                            colorbar_label="Pearson r",
                            title="Value (Ridge)  (white = r=0)")
fig.suptitle(
    f"Inference decoding  ·  stim period  ·  {pooling} pooling  [plasticity off]",
    y=1.04,
)
plt.tight_layout()
plt.show()

# ── figure 2: binary SVM by stimulus group ────────────────────────────────
fig, axes = plt.subplots(1, len(stim_groups),
                         figsize=(cell_size * n_contexts * len(stim_groups) + 1,
                                  cell_size * n_contexts),
                         squeeze=False)
for ax, (gname, gm) in zip(axes[0], infer_bin_gm.items()):
    plot_generalisation_heatmap(ax, gm, contexts,
                                vmin=0, vmax=1, cmap="seismic",
                                colorbar_label="Accuracy",
                                title=f"Binary SVM — {gname}  (chance = 0.5)")
fig.suptitle(
    f"Inference binary SVM  ·  stim period  ·  {pooling} pooling  [plasticity off]\n"
    f"low (v=0) vs high (v=1), all identities pooled within group",
    y=1.06,
)
plt.tight_layout()
plt.show()

## 8. Decoding — per context repetition (post-training activations)

In [ ]:
n_phases = int(activations["phase"].max()) + 1
phases   = np.arange(n_phases)

# stim_groups / group_colors defined in cell a0000021

within_stim_ph = np.full((n_phases, n_contexts), np.nan)
cross_stim_ph  = np.full((n_phases, n_contexts, n_contexts), np.nan)
within_val_ph  = np.full((n_phases, n_contexts), np.nan)
cross_val_ph   = np.full((n_phases, n_contexts, n_contexts), np.nan)
cross_val_grp  = {g: np.full((n_phases, n_contexts, n_contexts), np.nan) for g in stim_groups}
within_bin_grp = {g: np.full((n_phases, n_contexts), np.nan) for g in stim_groups}
cross_bin_grp  = {g: np.full((n_phases, n_contexts, n_contexts), np.nan) for g in stim_groups}

for p in range(n_phases):
    print(f"phase {p+1}/{n_phases}", end="\r")
    act_p = filter_act_dict(activations, activations["phase"] == p)

    acc_p = pairwise_decode(act_p, period="stim", pooling=pooling, n_folds=n_folds)
    for c in range(n_contexts):
        within_stim_ph[p, c] = mean_pairs(acc_p[c])

    cx_p = crosscontext_decode(act_p, period="stim", pooling=pooling)
    for ct in range(n_contexts):
        for ce in range(n_contexts):
            if ct != ce:
                cross_stim_ph[p, ct, ce] = mean_pairs(cx_p[ct, ce])

    within_val_ph[p] = value_decode_within(act_p, period="stim", pooling=pooling,
                                           value_matrix=value_matrix, n_folds=n_folds)
    r_crs = value_decode_cross(act_p, period="stim", pooling=pooling, value_matrix=value_matrix)
    cross_val_ph[p] = r_crs
    cross_val_grp["all"][p] = r_crs

    for gname, gidx in stim_groups.items():
        act_g = (filter_act_dict(act_p, np.isin(act_p["stimulus"], gidx))
                 if gidx is not None else act_p)
        if act_g["stimulus"].shape[0] < 2:
            continue
        if gname != "all":
            cross_val_grp[gname][p] = value_decode_cross(act_g, period="stim", pooling=pooling,
                                                          value_matrix=value_matrix)
        within_bin_grp[gname][p] = binary_value_decode_within(act_g, period="stim", pooling=pooling,
                                                               value_matrix=value_matrix, n_folds=n_folds)
        cross_bin_grp[gname][p]  = binary_value_decode_cross(act_g,  period="stim", pooling=pooling,
                                                              value_matrix=value_matrix)
print("done")

mean_cross_val_grp = {g: np.array([mean_offdiag(cross_val_grp[g][p]) for p in range(n_phases)])
                      for g in stim_groups}
mean_cross_bin_grp = {g: np.array([mean_offdiag(cross_bin_grp[g][p]) for p in range(n_phases)])
                      for g in stim_groups}

# ── figure 1: stim / Ridge value / binary SVM time-series ──────────────────
fig, axes = plt.subplots(3, 2, figsize=(12, 9), sharex=True)

for c in range(n_contexts):
    axes[0, 0].plot(phases, within_stim_ph[:, c], color=ctx_colors[c],
                    label=contexts[c], marker="o", ms=4)
axes[0, 0].axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
axes[0, 0].set_ylabel("Mean pairwise accuracy")
axes[0, 0].set_title("Stim identity — within context")
axes[0, 0].legend(fontsize=8); axes[0, 0].set_ylim(0, 1)

for ct in range(n_contexts):
    for ce in range(n_contexts):
        if ct != ce:
            axes[0, 1].plot(phases, cross_stim_ph[:, ct, ce],
                            color=ctx_colors[ct], linewidth=1.2, marker="o", ms=3,
                            label=f"{contexts[ct]}→{contexts[ce]}")
axes[0, 1].axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
axes[0, 1].set_title("Stim identity — cross-context")
axes[0, 1].set_ylim(0, 1); axes[0, 1].legend(fontsize=8)

for c in range(n_contexts):
    axes[1, 0].plot(phases, within_val_ph[:, c], color=ctx_colors[c],
                    label=contexts[c], marker="o", ms=4)
axes[1, 0].axhline(0, color="gray", linestyle="--", linewidth=0.8)
axes[1, 0].set_ylabel("Pearson r")
axes[1, 0].set_title("Value (Ridge) — within context")
axes[1, 0].set_ylim(-1, 1); axes[1, 0].legend(fontsize=8)

for ct in range(n_contexts):
    for ce in range(n_contexts):
        if ct != ce:
            axes[1, 1].plot(phases, cross_val_ph[:, ct, ce],
                            color=ctx_colors[ct], linewidth=1.2, marker="o", ms=3,
                            label=f"{contexts[ct]}→{contexts[ce]}")
axes[1, 1].axhline(0, color="gray", linestyle="--", linewidth=0.8)
axes[1, 1].set_title("Value (Ridge) — cross-context (all stim)")
axes[1, 1].set_ylim(-1, 1); axes[1, 1].legend(fontsize=8)

for gname, gc in group_colors.items():
    axes[2, 0].plot(phases, within_bin_grp[gname].mean(1), color=gc,
                    linewidth=1.5, marker="o", ms=4, label=gname)
axes[2, 0].axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
axes[2, 0].set_ylabel("Accuracy")
axes[2, 0].set_xlabel("Context repetition")
axes[2, 0].set_title("Binary SVM — within context (mean across contexts)")
axes[2, 0].set_ylim(0, 1); axes[2, 0].legend(fontsize=8)

for gname, gc in group_colors.items():
    axes[2, 1].plot(phases, mean_cross_bin_grp[gname], color=gc, linewidth=1.8,
                    marker="o", ms=5, label=gname, zorder=3)
axes[2, 1].axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
axes[2, 1].set_title("Binary SVM — cross-context (mean direction)")
axes[2, 1].set_xlabel("Context repetition")
axes[2, 1].set_ylim(0, 1); axes[2, 1].legend(fontsize=8)

for ax in axes.flat:
    ax.set_xticks(phases)
fig.suptitle(
    f"Decoding by context repetition  ·  stim period  ·  {pooling} pooling  (post-training)",
    y=1.01,
)
plt.tight_layout()
plt.show()

# ── figure 2: Ridge cross-context value by group + bar ────────────────────
bar_vals = {}
for gname, gidx in stim_groups.items():
    act_g = (filter_act_dict(activations, np.isin(activations["stimulus"], gidx))
             if gidx is not None else activations)
    r_g = value_decode_cross(act_g, period="stim", pooling=pooling, value_matrix=value_matrix)
    bar_vals[gname] = mean_offdiag(r_g)

fig, (ax_ts, ax_bar) = plt.subplots(1, 2, figsize=(12, 4.5),
                                     gridspec_kw={"width_ratios": [3, 1]})
for gname, gc in group_colors.items():
    ax_ts.plot(phases, mean_cross_val_grp[gname], color=gc, linewidth=1.8,
               marker="o", ms=5, label=gname, zorder=3)
    for ct in range(n_contexts):
        for ce in range(n_contexts):
            if ct != ce:
                ax_ts.plot(phases, cross_val_grp[gname][:, ct, ce],
                           color=gc, alpha=0.25, linewidth=0.8, linestyle="--")
ax_ts.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax_ts.axhline(id_val_r, color="gray", linestyle=":", linewidth=0.8,
              label=f"identity baseline (r={id_val_r:.2f})")
ax_ts.set_xlabel("Context repetition"); ax_ts.set_ylabel("Mean cross-context Pearson r")
ax_ts.set_title("Cross-context value generalisation (Ridge) by stim group")
ax_ts.set_ylim(-1, 1); ax_ts.set_xticks(phases); ax_ts.legend(fontsize=8)

bars = ax_bar.bar(list(stim_groups), [bar_vals[g] for g in stim_groups],
                  color=[group_colors[g] for g in stim_groups], width=0.5)
ax_bar.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax_bar.axhline(id_val_r, color="gray", linestyle=":", linewidth=0.8)
ax_bar.set_ylim(-1, 1); ax_bar.set_ylabel("Mean cross-context Pearson r")
ax_bar.set_title("Post-training summary\n(all phases pooled)")
for bar, val in zip(bars, [bar_vals[g] for g in stim_groups]):
    if not np.isnan(val):
        va, yoff = ("bottom", 0.03) if val >= 0 else ("top", -0.03)
        ax_bar.text(bar.get_x() + bar.get_width() / 2, val + yoff,
                    f"{val:.2f}", ha="center", va=va, fontsize=9)
plt.tight_layout()
plt.show()

# ── figure 3: binary SVM cross-context by group + bar ─────────────────────
bar_bin = {}
for gname, gidx in stim_groups.items():
    act_g = (filter_act_dict(activations, np.isin(activations["stimulus"], gidx))
             if gidx is not None else activations)
    bc = binary_value_decode_cross(act_g, period="stim", pooling=pooling, value_matrix=value_matrix)
    bar_bin[gname] = mean_offdiag(bc)

fig, (ax_ts3, ax_bar3) = plt.subplots(1, 2, figsize=(12, 4.5),
                                       gridspec_kw={"width_ratios": [3, 1]})
for gname, gc in group_colors.items():
    ax_ts3.plot(phases, mean_cross_bin_grp[gname], color=gc, linewidth=1.8,
                marker="o", ms=5, label=gname, zorder=3)
    for ct in range(n_contexts):
        for ce in range(n_contexts):
            if ct != ce:
                ax_ts3.plot(phases, cross_bin_grp[gname][:, ct, ce],
                            color=gc, alpha=0.25, linewidth=0.8, linestyle="--")
ax_ts3.axhline(0.5, color="gray", linestyle="--", linewidth=0.8, label="chance")
ax_ts3.set_xlabel("Context repetition"); ax_ts3.set_ylabel("Accuracy")
ax_ts3.set_title("Binary SVM — cross-context by stim group  (low vs high, identities pooled)")
ax_ts3.set_ylim(0, 1); ax_ts3.set_xticks(phases); ax_ts3.legend(fontsize=8)

bars = ax_bar3.bar(list(stim_groups), [bar_bin[g] for g in stim_groups],
                   color=[group_colors[g] for g in stim_groups], width=0.5)
ax_bar3.axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
ax_bar3.set_ylim(0, 1); ax_bar3.set_ylabel("Accuracy")
ax_bar3.set_title("Post-training summary\n(all phases pooled)")
for bar, val in zip(bars, [bar_bin[g] for g in stim_groups]):
    if not np.isnan(val):
        ax_bar3.text(bar.get_x() + bar.get_width() / 2, val + 0.02,
                     f"{val:.2f}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

## 9. Decoding — per context repetition (training-time activations)

In [ ]:
stim_hidden_tr   = np.stack([hidden_np_train[t["stim_window"][0]   : t["stim_window"][1]]   for t in state_seq.trial_structure])
reward_hidden_tr = np.stack([hidden_np_train[t["reward_window"][0] : t["reward_window"][1]] for t in state_seq.trial_structure])

_cc_tr = defaultdict(int)
phase_arr_tr = np.zeros(n_trials, dtype=int)
for i, ctx in enumerate(context_arr):
    phase_arr_tr[i] = _cc_tr[ctx] // trials_per_phase
    _cc_tr[ctx] += 1

activations_train = {
    "hidden_states":    hidden_np_train,
    "stim_hidden":      stim_hidden_tr,
    "reward_hidden":    reward_hidden_tr,
    "context":          context_arr,
    "stimulus":         stimulus_arr,
    "phase":            phase_arr_tr,
    "reward_available": reward_available_arr,
    "trial_structure":  state_seq.trial_structure,
}

n_phases_tr = int(phase_arr_tr.max()) + 1
cell_size   = 1.1

fig, axes = plt.subplots(
    2, n_phases_tr,
    figsize=(cell_size * n_contexts * n_phases_tr + 1, cell_size * n_contexts * 2 + 1),
    squeeze=False,
)

for p in range(n_phases_tr):
    print(f"  phase {p+1}/{n_phases_tr}", end="\r")
    act_p = filter_act_dict(activations_train, phase_arr_tr == p)

    acc_p  = pairwise_decode(act_p, period="stim", pooling=pooling, n_folds=n_folds)
    cx_p   = crosscontext_decode(act_p, period="stim", pooling=pooling)
    gm_s_p = generalisation_matrix(acc_p, cx_p)

    r_win_p = value_decode_within(act_p, period="stim", pooling=pooling,
                                  value_matrix=value_matrix, n_folds=n_folds)
    r_crs_p = value_decode_cross(act_p, period="stim", pooling=pooling,
                                 value_matrix=value_matrix)
    gm_v_p  = value_gen_matrix(r_win_p, r_crs_p)

    for row, (gm, vmin, vmax, clabel, ylabel) in enumerate([
        (gm_s_p, 0,  1,  "Accuracy",  "Stim\nTrain ctx"),
        (gm_v_p, -1, 1,  "Pearson r", "Value\nTrain ctx"),
    ]):
        plot_generalisation_heatmap(
            axes[row, p], gm, contexts,
            vmin=vmin, vmax=vmax, cmap="seismic",
            colorbar_label=clabel,
            title=f"Rep {p+1}" if row == 0 else None,
            ylabel=ylabel if p == 0 else "Train ctx",
        )

print("done")
fig.suptitle(
    f"Training-time decoding by context repetition  ·  stim period  ·  {pooling} pooling\n"
    f"Row 1: stim generalisation (white = chance)  ·  Row 2: value decoding (white = r=0)",
    y=1.02, fontsize=10,
)
plt.tight_layout()
plt.show()

In [ ]:
# ── training-time cross-context value generalisation by stimulus group ──────
# stim_groups defined in cell a0000023: all / swap=[0,2,3,4] / anchor=[5,6,7,8]
cross_val_tr_grp = {g: np.full((n_phases_tr, n_contexts, n_contexts), np.nan) for g in stim_groups}

for p in range(n_phases_tr):
    print(f"phase {p+1}/{n_phases_tr}", end="\r")
    act_p = filter_act_dict(activations_train, phase_arr_tr == p)

    cross_val_tr_grp["all"][p] = value_decode_cross(act_p, period="stim", pooling=pooling,
                                                     value_matrix=value_matrix)
    for gname, gidx in stim_groups.items():
        if gname == "all":
            continue
        gmask = np.isin(act_p["stimulus"], gidx)
        if gmask.sum() < 2:
            continue
        act_g = filter_act_dict(act_p, gmask)
        cross_val_tr_grp[gname][p] = value_decode_cross(act_g, period="stim", pooling=pooling,
                                                          value_matrix=value_matrix)
print("done")

mean_cross_val_tr_grp = {g: np.array([mean_offdiag(cross_val_tr_grp[g][p]) for p in range(n_phases_tr)])
                          for g in stim_groups}

# training-time summary bar: full activations_train pooled
bar_vals_tr = {}
for gname, gidx in stim_groups.items():
    act_g = (filter_act_dict(activations_train, np.isin(activations_train["stimulus"], gidx))
             if gidx is not None else activations_train)
    r_g = value_decode_cross(act_g, period="stim", pooling=pooling, value_matrix=value_matrix)
    bar_vals_tr[gname] = mean_offdiag(r_g)

phases_tr = np.arange(n_phases_tr)

fig, (ax_ts, ax_bar) = plt.subplots(1, 2, figsize=(12, 4.5),
                                     gridspec_kw={"width_ratios": [3, 1]})

for gname, gc in group_colors.items():
    ax_ts.plot(phases_tr, mean_cross_val_tr_grp[gname], color=gc, linewidth=1.8,
               marker="o", ms=5, label=gname, zorder=3)
    for ct in range(n_contexts):
        for ce in range(n_contexts):
            if ct != ce:
                ax_ts.plot(phases_tr, cross_val_tr_grp[gname][:, ct, ce],
                           color=gc, alpha=0.25, linewidth=0.8, linestyle="--")

ax_ts.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax_ts.axhline(id_val_r, color="gray", linestyle=":", linewidth=0.8,
              label=f"identity baseline (r={id_val_r:.2f})")
ax_ts.set_xlabel("Context repetition (training)")
ax_ts.set_ylabel("Mean cross-context Pearson r")
ax_ts.set_title("Cross-context value generalisation by stimulus group  [training-time]")
ax_ts.set_ylim(-1, 1)
ax_ts.set_xticks(phases_tr)
ax_ts.legend(fontsize=8)

bars = ax_bar.bar(list(stim_groups), [bar_vals_tr[g] for g in stim_groups],
                  color=[group_colors[g] for g in stim_groups], width=0.5)
ax_bar.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax_bar.axhline(id_val_r, color="gray", linestyle=":", linewidth=0.8)
ax_bar.set_ylim(-1, 1)
ax_bar.set_ylabel("Mean cross-context Pearson r")
ax_bar.set_title("Training-time summary\n(all phases pooled)")
for bar, val in zip(bars, [bar_vals_tr[g] for g in stim_groups]):
    if not np.isnan(val):
        va, yoff = ("bottom", 0.03) if val >= 0 else ("top", -0.03)
        ax_bar.text(bar.get_x() + bar.get_width() / 2, val + yoff,
                    f"{val:.2f}", ha="center", va=va, fontsize=9)
plt.tight_layout()
plt.show()

## 10. Save

In [ ]:
run_id = (
    f"RNN_9s2c_swap_anchors"
    f"_h{hidden_size}_g{recurrent_gain}"
    f"_ep{n_episodes}_bptt{bptt_len}_upd{update_every}"
    f"_seed{seed}"
    f"_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
)

run_dir = Path("../results") / run_id
run_dir.mkdir(parents=True, exist_ok=True)

with open(run_dir / "activations.pkl", "wb") as f:
    pickle.dump(activations, f, protocol=pickle.HIGHEST_PROTOCOL)
with open(run_dir / "activations_train.pkl", "wb") as f:
    pickle.dump(activations_train, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"run_id : {run_id}")
print(f"Saved  → {run_dir}")
# ── save vis_data.pkl for use with 13_05_26_vis_inference.ipynb ──────────
vis_data = {
    "value_matrix":          value_matrix,
    "n_stimuli":             n_stimuli,
    "n_contexts":            n_contexts,
    "stimuli":               stimuli,
    "contexts":              contexts,
    "stim_group_info":       stim_group_info,
    "stim_timesteps":        stim_timesteps,
    "reward_timesteps":      reward_timesteps,
    "reward_lick":           env.reward_lick,
    "reward_lick_fa":      env.reward_lick_fa,
    "reward_no_lick":        env.reward_no_lick,
    "lick_cost":             lick_cost,
    "infer_trial_data":      infer_trial_data,
    "infer_action_seq":      infer_action_seq,
    "infer_trial_structure": infer_state_seq.trial_structure,
}
with open(run_dir / "vis_data.pkl", "wb") as f:
    pickle.dump(vis_data, f, protocol=pickle.HIGHEST_PROTOCOL)
print(f"vis_data saved \u2192 {run_dir / 'vis_data.pkl'}")
